In [2]:
import os
import pandas as pd
from pathlib import Path

In [3]:
outcome_df_path = ''
outcome_df = pd.read_csv(outcome_df_path)
outcome_df['date'] = pd.to_datetime(outcome_df['date'])
outcome_df['Time Stamp (seconds)'] = outcome_df['Time Stamp (seconds)'].round()
outcome_df.sort_values(by=['subject_id', 'date', 'Time Stamp (seconds)'], inplace=True)

# remove first 2 hours of data to replicate other paper
outcome_df = outcome_df.loc[outcome_df['Time Stamp (seconds)'] >= 7200].copy()


In [4]:
outcome_df['unique_id'] = outcome_df['subject_id'] + '__' + outcome_df['date'].astype(str)
outcome_df['hypotension_replaced'] = outcome_df['hypotension'].replace({2:0})
outcome_df['rolling_10min_hypotension'] = outcome_df.groupby('unique_id')['hypotension_replaced'].transform(lambda x: x.rolling(10, step=1, center=False).sum())

outcome_df.dropna(subset=['rolling_10min_hypotension'], inplace=True)

outcome_df['hypotension_5min_in_10min_win'] = (outcome_df['rolling_10min_hypotension'] >= 5).astype(int)

subjects_with_hypotension = outcome_df.groupby('unique_id')['hypotension_5min_in_10min_win'].any()
non_hypotension_subjects = subjects_with_hypotension[~subjects_with_hypotension].index
hypotension_subjects = subjects_with_hypotension[subjects_with_hypotension].index

outcome_df_hypotension = outcome_df.loc[outcome_df['unique_id'].isin(hypotension_subjects)].copy()
outcome_df_non_hypotension = outcome_df.loc[outcome_df['unique_id'].isin(non_hypotension_subjects)].copy()


# Get the first occurrence of hypotension for each subject
first_hypotension = outcome_df_hypotension.groupby('unique_id').apply(
    lambda x: x.index[x['hypotension_5min_in_10min_win'] == 1].min()
).to_dict()

# Select rows up to and including first hypotension for each subject
outcome_df_hypotension = outcome_df_hypotension.groupby('unique_id').apply(
    lambda x: x.loc[first_hypotension[x.name]]
).reset_index(level=0, drop=True)

outcome_df_hypotension['int_subject_id'] = outcome_df_hypotension['subject_id'].str.strip('p').astype(int)
outcome_df_hypotension['current_time'] = outcome_df_hypotension['date'] + pd.to_timedelta(outcome_df_hypotension['Time Stamp (seconds)'].astype(int), unit='s')

outcome_df_non_hypotension['int_subject_id'] = outcome_df_non_hypotension['subject_id'].str.strip('p').astype(int)
outcome_df_non_hypotension['current_time'] = outcome_df_non_hypotension['date'] + pd.to_timedelta(outcome_df_non_hypotension['Time Stamp (seconds)'].astype(int), unit='s')


In [12]:

final_outcome_hypotension_df = outcome_df_hypotension.copy()
final_outcome_hypotension_df['hypotension'] = 1 # all are 1!


In [13]:
# subtract 15 min from hypotension event.
mean_time_to_hypotension = final_outcome_hypotension_df['Time Stamp (seconds)'].mean()
final_outcome_hypotension_df.sort_values(by=['unique_id', 'Time Stamp (seconds)'], inplace=True)

In [14]:
outcome_df_non_hypotension_mt = outcome_df_non_hypotension.loc[outcome_df_non_hypotension['Time Stamp (seconds)'] <= mean_time_to_hypotension].copy()

# filter those who died / ages
final_outcome_df_non_hypotension = outcome_df_non_hypotension_mt.copy()

final_outcome_df_non_hypotension.hypotension = 0 # all are 0!
final_outcome_df_non_hypotension.sort_values(by=['unique_id', 'Time Stamp (seconds)'], inplace=True)

In [15]:
# get the last non-hypotension event for each subject
final_outcome_df_non_hypotension = final_outcome_df_non_hypotension.groupby('unique_id', as_index=False).apply(lambda x: x.iloc[-1])
final_outcome_df_non_hypotension.drop_duplicates(subset=['unique_id'], inplace=True)


In [18]:
relv_cols = ['SBP (mmHg)', 'DBP (mmHg)', 'MBP (mmHg)', 'hypotension',
       'Time Stamp (seconds)', 'subject_id', 'date']

In [ ]:
final_df = pd.concat([final_outcome_df_non_hypotension[relv_cols], final_outcome_hypotension_df[relv_cols]])
final_df['unique_identifier'] = final_df['subject_id'].astype(str) + '__' + final_df['date'].astype(str)
final_df.to_csv('', index=False)